In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print("GPU Memory:", round(gpu_memory, 2), "GB")
else:
    print("GPU not detected. Please check Runtime settings.")

PyTorch Version: 2.11.0+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


In [2]:
!pip install -q unsloth sentence-transformers faiss-cpu pypdf

In [3]:
import torch
from unsloth import FastLanguageModel

# Model configuration
model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"

# Load quantized model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

print("Model loaded successfully!")
print("Model:", model_name)
print("4-bit Quantization: Enabled")
print("Tokenizer loaded successfully!")

print(
    "GPU Memory Used:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.11: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded successfully!
Model: unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit
4-bit Quantization: Enabled
Tokenizer loaded successfully!
GPU Memory Used: 2.2 GB


In [4]:
from google.colab import files
from pypdf import PdfReader
import os

# Upload PDF
uploaded = files.upload()

# Read uploaded PDF
documents = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        documents.append({
            "filename": filename,
            "text": text
        })

        print("PDF Loaded:", filename)
        print("Total Pages:", len(reader.pages))
        print("Extracted Characters:", len(text))

print("\nTotal Documents:", len(documents))

Saving ragpdf.pdf to ragpdf.pdf
PDF Loaded: ragpdf.pdf
Total Pages: 19
Extracted Characters: 69063

Total Documents: 1


In [5]:
# Text Chunking

chunk_size = 1000
chunk_overlap = 200

chunks = []

for document in documents:

    text = document["text"]

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append({
                "text": chunk,
                "source": document["filename"]
            })

        start += chunk_size - chunk_overlap

print("Total Documents:", len(documents))
print("Total Chunks:", len(chunks))

print("\nFirst Chunk Preview:\n")
print(chunks[0]["text"][:500])

Total Documents: 1
Total Chunks: 87

First Chunk Preview:

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁ


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

# Extract text from chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Convert embeddings to float32
embeddings = embeddings.astype("float32")

print("\nEmbeddings generated successfully!")
print("Embedding Shape:", embeddings.shape)

# Create FAISS vector index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add embeddings to FAISS
index.add(embeddings)

print("\nFAISS Vector Database Created!")
print("Total Indexed Vectors:", index.ntotal)
print("Vector Dimension:", dimension)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Embeddings generated successfully!
Embedding Shape: (87, 384)

FAISS Vector Database Created!
Total Indexed Vectors: 87
Vector Dimension: 384


In [7]:
def retrieve_documents(query, top_k=3):

    # Convert user question into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS database
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):

        if idx != -1:

            retrieved_chunks.append({
                "text": chunks[idx]["text"],
                "source": chunks[idx]["source"],
                "similarity_score": float(score)
            })

    return retrieved_chunks


# Test retrieval
query = "What is Retrieval-Augmented Generation?"

results = retrieve_documents(query, top_k=3)

print("User Question:", query)
print("\nRetrieved Documents:\n")

for i, result in enumerate(results, 1):

    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Similarity:", round(result["similarity_score"], 4))
    print("Content:", result["text"][:500])

User Question: What is Retrieval-Augmented Generation?

Retrieved Documents:


--- Result 1 ---
Source: ragpdf.pdf
Similarity: 0.5947
Content: item, but on aggregating content from several
pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6 Discussion
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that pe

--- Result 2 ---
Source: ragpdf.pdf
Similarity: 0.553
Content: t consist of trainable parameters, but does consists of 21M
728 dimensional vectors, consisting of 15.3B values. These can be easily be stored at 8-bit ﬂoating
point precision to manage memory and disk footprints.
H Retrieval Collapse
In preliminary experiments, we observed that for some tas

In [8]:
from unsloth import FastLanguageModel
import torch

# Enable optimized inference
FastLanguageModel.for_inference(model)

def rag_generate(query, top_k=3):

    # Step 1: Retrieve relevant documents
    retrieved_docs = retrieve_documents(query, top_k=top_k)

    # Step 2: Prepare context
    context = "\n\n".join([
        doc["text"] for doc in retrieved_docs
    ])

    # Step 3: Create RAG prompt
    messages = [
        {
            "role": "system",
            "content": """You are a helpful AI research assistant.

Answer the user's question using ONLY the provided context.

Rules:
1. Use the retrieved context as your primary source.
2. Do not invent facts or information.
3. If the answer is not available in the context,
   clearly say that the provided documents do not contain
   enough information to answer.
4. Give clear and understandable answers."""
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{query}

Answer:
"""
        }
    ]

    # Step 4: Tokenize prompt
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Step 5: Generate answer
    with torch.inference_mode():

        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Step 6: Decode generated answer
    input_length = inputs.shape[1]

    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return answer, retrieved_docs


print("RAG Generation Pipeline Ready!")

RAG Generation Pipeline Ready!


In [9]:
question = "What is Retrieval-Augmented Generation?"

answer, sources = rag_generate(question)

print("USER QUESTION:")
print(question)

print("\nAI GENERATED ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print("-", source["source"])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


USER QUESTION:
What is Retrieval-Augmented Generation?

AI GENERATED ANSWER:
The provided context does not explicitly define what "Retrieval-Augmented Generation" (RAG) is. However, based on the context, it appears that RAG is a type of hybrid generation model that combines parametric and non-parametric memory, and is used for open-domain question answering (QA) and other tasks. It seems to be a variant of the RAG model, which is a type of model that uses a retrieval component to learn relevant information for the task.

SOURCES:
- ragpdf.pdf
- ragpdf.pdf
- ragpdf.pdf


In [10]:
# Improved paragraph-aware chunking

import re

def improved_chunking(text, chunk_size=1500, overlap=200):

    paragraphs = [
        p.strip() for p in re.split(r'\n+', text)
        if p.strip()
    ]

    result = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(paragraph) > chunk_size:

            if current_chunk:
                result.append(current_chunk)
                current_chunk = ""

            start = 0

            while start < len(paragraph):
                result.append(paragraph[start:start + chunk_size])
                start += chunk_size - overlap

        elif len(current_chunk) + len(paragraph) <= chunk_size:

            current_chunk += "\n" + paragraph

        else:

            result.append(current_chunk)

            current_chunk = (
                current_chunk[-overlap:] + "\n" + paragraph
            )

    if current_chunk:
        result.append(current_chunk)

    return result


# Rebuild chunks
chunks = []

for document in documents:

    new_chunks = improved_chunking(document["text"])

    for chunk in new_chunks:

        chunks.append({
            "text": chunk,
            "source": document["filename"]
        })

print("Improved Chunking Completed!")
print("Total Improved Chunks:", len(chunks))
print("\nFirst Chunk Preview:")
print(chunks[0]["text"][:500])

Improved Chunking Completed!
Total Improved Chunks: 55

First Chunk Preview:

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when 


In [11]:
# Generate improved embeddings

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# Rebuild FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Updated FAISS Index Created!")
print("Total Vectors:", index.ntotal)
print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Updated FAISS Index Created!
Total Vectors: 55
Embedding Shape: (55, 384)


In [12]:
test_questions = [
    "What is Retrieval-Augmented Generation?",
    "What is parametric memory?",
    "What is non-parametric memory?",
    "What are the advantages of RAG models?"
]

for question in test_questions:

    answer, sources = rag_generate(question)

    print("\n" + "=" * 60)
    print("QUESTION:", question)
    print("\nANSWER:", answer)
    print("\nSOURCE:", sources[0]["source"])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is Retrieval-Augmented Generation?

ANSWER: The provided context does not explicitly define "Retrieval-Augmented Generation." However, based on the context, it appears that "Retrieval-Augmented Generation" is related to the RAG (Retrieval-Augmented Generation) model, which is a type of non-parametric memory model that combines retrieval and generation capabilities.

In the context, RAG is mentioned as a model that uses a dense retriever to learn relevant information for the task, and it is compared to other models such as BART and BM25. The model is also mentioned in the context of retrieval collapse, where the retrieval component learns to retrieve the same documents regardless of the input.

Therefore, it can be inferred that "Retrieval-Augmented Generation" refers to the RAG model, which combines retrieval and generation capabilities to generate text.

SOURCE: ragpdf.pdf


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is parametric memory?

ANSWER: Unfortunately, the provided context does not contain enough information to answer the question about what parametric memory is. The context only mentions non-parametric memory, which is a retrieval-based memory, and discusses the benefits of combining parametric and non-parametric memory components.

SOURCE: ragpdf.pdf


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is non-parametric memory?

ANSWER: According to the provided context, non-parametric memory is a dense vector index of Wikipedia, accessed with a pre-trained neural retriever.

SOURCE: ragpdf.pdf

QUESTION: What are the advantages of RAG models?

ANSWER: According to the provided context, the advantages of RAG models are:

1. RAG models hallucinate less and generate factually correct text more often than BART.
2. RAG generations are more diverse than BART generations.
3. RAG models can generate responses that combine content from several documents, which may perform best on Jeopardy question generation tasks.
4. RAG sets a new state-of-the-art on tasks like QA, without the need for expensive, specialized "salient span masking" pre-training.
5. RAG demonstrates that neither a re-ranker nor extractive reader is necessary for state-of-the-art performance.
6. RAG models can generate answers even when it is possible to extract them, as documents with clues about the answer b

In [13]:
print("GPU:", torch.cuda.get_device_name(0))

print(
    "Allocated Memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "Reserved Memory:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print(
    "Total GPU Memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

GPU: Tesla T4
Allocated Memory: 2.55 GB
Reserved Memory: 2.81 GB
Total GPU Memory: 14.56 GB


In [14]:
question = """What are parametric and non-parametric memory
in Retrieval-Augmented Generation, and how do they work together?"""

answer, sources = rag_generate(question)

print("USER QUESTION:")
print(question)

print("\nAI GENERATED ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, source in enumerate(sources, 1):
    print(f"\nSource {i}: {source['source']}")
    print("Similarity:", round(source["similarity_score"], 4))
    print(source["text"][:500])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION:
What are parametric and non-parametric memory
in Retrieval-Augmented Generation, and how do they work together?

AI GENERATED ANSWER:
According to the provided context, in Retrieval-Augmented Generation (RAG), "parametric" and "non-parametric" memory refer to two types of memory components used in the model.

"Parametric" memory refers to a component that is trained as a part of the model, using a parametric objective function, such as a generator. This type of memory is learned through training and is typically a key component of the model.

"Non-parametric" memory, on the other hand, refers to a component that is pre-trained and pre-loaded with extensive knowledge, without requiring additional training. This type of memory is not trainable and consists of a large set of vectors (21M, 728-dimensional) that can be easily stored and retrieved.

In RAG, both parametric and non-parametric memory components work together to generate responses to input queries. The parametric

In [15]:
question = "Who won the FIFA World Cup in 2022?"

answer, sources = rag_generate(question)

print("USER QUESTION:")
print(question)

print("\nAI GENERATED ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for i, source in enumerate(sources, 1):
    print(f"Source {i}: {source['source']}")
    print("Similarity:", round(source["similarity_score"], 4))

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER QUESTION:
Who won the FIFA World Cup in 2022?

AI GENERATED ANSWER:
The provided context does not contain enough information to answer the question "Who won the FIFA World Cup in 2022?"

RETRIEVED SOURCES:
Source 1: ragpdf.pdf
Similarity: 0.1634
Source 2: ragpdf.pdf
Similarity: 0.1487
Source 3: ragpdf.pdf
Similarity: 0.1363


In [16]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU Memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("Allocated GPU Memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("Reserved GPU Memory:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

print("\nModel Configuration:")
print("Model:", model_name)
print("4-bit Loading:", True)

print("\nModel Parameter Data Types:")
dtypes = {}

for parameter in model.parameters():
    dtype = str(parameter.dtype)
    dtypes[dtype] = dtypes.get(dtype, 0) + parameter.numel()

for dtype, count in dtypes.items():
    print(dtype, ":", count, "parameters")

GPU: Tesla T4
Total GPU Memory: 14.56 GB
Allocated GPU Memory: 2.53 GB
Reserved GPU Memory: 2.81 GB

Model Configuration:
Model: unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit
4-bit Loading: True

Model Parameter Data Types:
torch.float16 : 469675008 parameters
torch.uint8 : 1371537408 parameters


In [17]:
print("QUANTIZATION CONFIGURATION")

print("Model config:", model.config)

print("\nQUANTIZED MODULES:")

count = 0

for name, module in model.named_modules():
    if "Linear4bit" in str(type(module)):
        print(name, ":", type(module).__name__)
        count += 1

print("\nTotal 4-bit Linear Modules:", count)

QUANTIZATION CONFIGURATION
Model config: LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": [
      "lm_head",
      "multi_modal_projector",
      "merger",
      "modality_projection",
      "model.layers.1.mlp"
    ],
    "llm_int8_threshold": 6.0,
    "load_i